# Divorce Analysis — Data Preparation

**Changes in Socioeconomic Characteristics Associated with Recent Divorce in the United States in 2008, 2016 & 2024**

This notebook turns the raw IPUMS USA extract (`usa_00008`) into an analysis-ready CSV:

1. **Parse** the IPUMS DDI codebook (`usa_00008.xml`) to get each variable's fixed-width position, implied decimals, and value labels.
2. **Decode** the fixed-width data file (`usa_00008.dat` or `usa_00008.dat.gz`) into `usa_00008_decoded.csv`, adding a `*_LABEL` column for every categorical variable.
3. **Clean** the decoded file by dropping technical fields and redundant code/label columns → `usa_00008_cleaned.csv`.

### Required input files

| File | Where to get it |
|---|---|
| `usa_00008.xml` | In this repository |
| `usa_00008.dat.gz` (or the unzipped `usa_00008.dat`) | IPUMS USA extract download (not committed to git — too large) |

### Running on Google Colab
Upload both files to a folder in Google Drive (default: `MyDrive/divorce-analysis/`) and run all cells. The setup cell mounts Drive automatically. Edit `COLAB_DATA_DIR` below if you use a different folder.

### Running on Kaggle
Upload both files as a Kaggle Dataset and attach it to the notebook (**Add Input**). The setup cell searches `/kaggle/input` for them automatically. Outputs are written to `/kaggle/working`.

### Running locally
Put the files next to this notebook.

## 0. Setup and environment detection

In [6]:
import gzip
import os
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
IN_KAGGLE = os.path.exists("/kaggle/input")

# Folder in Google Drive that holds usa_00008.xml and usa_00008.dat(.gz)
COLAB_DATA_DIR = "/content/drive/MyDrive/divorce-analysis"

XML_NAME = "usa_00008.xml"
DAT_NAMES = ["usa_00008.dat", "usa_00008.dat.gz"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(COLAB_DATA_DIR)
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    # Find the attached dataset that contains the codebook
    matches = list(Path("/kaggle/input").rglob(XML_NAME))
    if not matches:
        raise FileNotFoundError(f"{XML_NAME} not found under /kaggle/input — attach the dataset first.")
    DATA_DIR = matches[0].parent
    WORK_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path.cwd()
    WORK_DIR = Path.cwd()

XML_FILE = DATA_DIR / XML_NAME

# Prefer the uncompressed .dat (faster), fall back to .dat.gz
DAT_FILE = next(
    (DATA_DIR / n for n in DAT_NAMES if (DATA_DIR / n).exists()),
    None,
)

DECODED_FILE = WORK_DIR / "usa_00008_decoded.csv"
CLEANED_FILE = WORK_DIR / "usa_00008_cleaned.csv"

if not XML_FILE.exists():
    raise FileNotFoundError(f"Missing codebook: {XML_FILE}")
if DAT_FILE is None:
    raise FileNotFoundError(f"Missing data file: one of {DAT_NAMES} in {DATA_DIR}")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"Codebook:    {XML_FILE}")
print(f"Data file:   {DAT_FILE} ({DAT_FILE.stat().st_size / 1024**2:,.0f} MB)")
print(f"Decoded out: {DECODED_FILE}")
print(f"Cleaned out: {CLEANED_FILE}")

Environment: Local
pandas 3.0.6, numpy 2.4.3
Codebook:    c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008.xml
Data file:   c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008.dat (947 MB)
Decoded out: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_decoded.csv
Cleaned out: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_cleaned.csv


## 1. Parse the IPUMS DDI codebook (XML)

Each `<var>` element gives the variable's column positions in the fixed-width file (`StartPos`/`EndPos`), the number of implied decimals (`dcml`), and — for categorical variables — the code → label mapping.

In [7]:
print("Reading XML metadata...")

tree = ET.parse(XML_FILE)
root = tree.getroot()

ns = {"ddi": "ddi:codebook:2_5"}

variables = []

for var in root.findall(".//ddi:dataDscr/ddi:var", ns):

    name = var.get("name")

    loc = var.find("ddi:location", ns)

    start = int(loc.get("StartPos")) - 1
    end = int(loc.get("EndPos"))

    decimals = int(var.get("dcml", "0"))

    labels = {}

    for cat in var.findall("ddi:catgry", ns):

        value_node = cat.find("ddi:catValu", ns)
        label_node = cat.find("ddi:labl", ns)

        if value_node is None or value_node.text is None:
            continue

        value = value_node.text.strip()

        label = (
            label_node.text.strip()
            if label_node is not None and label_node.text
            else ""
        )

        labels[value] = label

    variables.append({
        "name": name,
        "start": start,
        "end": end,
        "width": end - start,
        "decimals": decimals,
        "labels": labels
    })


print(f"Found {len(variables)} IPUMS variables.")

Reading XML metadata...
Found 46 IPUMS variables.


In [8]:
# Codebook overview: layout of every variable in the fixed-width file
codebook = pd.DataFrame([
    {
        "name": v["name"],
        "start_pos": v["start"] + 1,
        "end_pos": v["end"],
        "width": v["width"],
        "decimals": v["decimals"],
        "n_labels": len(v["labels"]),
        "type": "categorical" if v["labels"] else "numeric",
    }
    for v in variables
])
codebook

,name,start_pos,end_pos,width,decimals,n_labels,type
0,YEAR,1,4,4,0,39,categorical
1,SAMPLE,5,10,6,0,154,categorical
2,SERIAL,11,18,8,0,0,numeric
3,CBSERIAL,19,31,13,0,0,numeric
4,HHWT,32,41,10,2,0,numeric
5,HHTYPE,42,42,1,0,9,categorical
6,CLUSTER,43,55,13,0,0,numeric
7,REGION,56,57,2,0,17,categorical
8,STATEFIP,58,59,2,0,62,categorical
9,METRO,60,60,1,0,5,categorical


## 2. Decode the fixed-width DAT file → CSV

The file is ~1 GB uncompressed (several million rows), so it is processed in chunks and appended to the output CSV. Categorical codes are kept as strings (to preserve leading zeros such as `01`) and a readable `*_LABEL` column is added next to each one. Numeric variables are converted to numbers and IPUMS implied decimals are applied.

In [9]:
# Increase if you have plenty of RAM.
# 250k is reasonable for ~16 GB RAM (Colab/Kaggle default sessions are fine).
CHUNK_SIZE = 250_000


def decode_chunk(lines):

    data = {}

    for var in variables:

        name = var["name"]
        start = var["start"]
        end = var["end"]

        values = [
            line[start:end].strip()
            for line in lines
        ]

        # Preserve raw codes initially
        data[name] = values

        # Add readable labels
        if var["labels"]:

            mapping = var["labels"]

            data[name + "_LABEL"] = [
                mapping.get(v, "")
                for v in values
            ]

    df = pd.DataFrame(data)

    # -----------------------------------------------------
    # Numeric conversion
    # -----------------------------------------------------

    for var in variables:

        name = var["name"]

        # Categorical codes remain as strings.
        # Important because codes such as 01 must retain
        # their leading zeros.
        if var["labels"]:
            continue

        df[name] = pd.to_numeric(
            df[name],
            errors="coerce"
        )

        # IPUMS implied decimals
        if var["decimals"] > 0:
            df[name] /= 10 ** var["decimals"]

    return df


def open_dat(path):
    # Read either the plain .dat or the gzipped download directly
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="ISO-8859-1")
    return open(path, "r", encoding="ISO-8859-1", buffering=1024 * 1024 * 16)

In [10]:
start_time = time.time()

rows_processed = 0
first_chunk = True


def write_chunk(batch):
    global rows_processed, first_chunk

    df = decode_chunk(batch)

    df.to_csv(
        DECODED_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    rows_processed += len(batch)
    first_chunk = False


if DECODED_FILE.exists():
    DECODED_FILE.unlink()


with open_dat(DAT_FILE) as f:

    batch = []

    for line in f:

        batch.append(line.rstrip("\r\n"))

        if len(batch) == CHUNK_SIZE:

            write_chunk(batch)
            batch.clear()

            elapsed = time.time() - start_time

            print(
                f"{rows_processed:,} rows processed | "
                f"{rows_processed / elapsed:,.0f} rows/sec | "
                f"{elapsed / 60:.1f} min elapsed"
            )

    # Final partial chunk
    if batch:
        write_chunk(batch)


elapsed = time.time() - start_time

size_gb = DECODED_FILE.stat().st_size / (1024 ** 3)

print()
print("========================================")
print("Conversion complete")
print("========================================")
print(f"Rows:        {rows_processed:,}")
print(f"Variables:   {len(variables)}")
print(f"Output:      {DECODED_FILE}")
print(f"CSV size:    {size_gb:.2f} GB")
print(f"Runtime:     {elapsed / 60:.2f} minutes")
print(
    f"Average:     "
    f"{rows_processed / elapsed:,.0f} rows/sec"
)

250,000 rows processed | 30,228 rows/sec | 0.1 min elapsed
500,000 rows processed | 31,303 rows/sec | 0.3 min elapsed
750,000 rows processed | 31,777 rows/sec | 0.4 min elapsed
1,000,000 rows processed | 31,942 rows/sec | 0.5 min elapsed
1,250,000 rows processed | 32,127 rows/sec | 0.6 min elapsed
1,500,000 rows processed | 32,077 rows/sec | 0.8 min elapsed
1,750,000 rows processed | 32,133 rows/sec | 0.9 min elapsed
2,000,000 rows processed | 32,191 rows/sec | 1.0 min elapsed
2,250,000 rows processed | 32,255 rows/sec | 1.2 min elapsed
2,500,000 rows processed | 32,308 rows/sec | 1.3 min elapsed
2,750,000 rows processed | 32,351 rows/sec | 1.4 min elapsed
3,000,000 rows processed | 32,380 rows/sec | 1.5 min elapsed
3,250,000 rows processed | 32,370 rows/sec | 1.7 min elapsed
3,500,000 rows processed | 32,382 rows/sec | 1.8 min elapsed
3,750,000 rows processed | 32,375 rows/sec | 1.9 min elapsed
4,000,000 rows processed | 32,346 rows/sec | 2.1 min elapsed
4,250,000 rows processed | 32,

In [ ]:
# Preview of the decoded file
decoded_preview = pd.read_csv(DECODED_FILE, nrows=5, dtype=str, keep_default_na=False)
print(f"Decoded columns: {decoded_preview.shape[1]}")
decoded_preview

## 3. Clean features

For every **categorical** feature the decoder produced two columns: the numeric IPUMS code and its readable label. Only the numeric duplicate is removed, e.g. `DIVINYR` is dropped and `DIVINYR_LABEL` is kept. All other columns are left unchanged.

Variables that are real numbers but also carry IPUMS labels (`YEAR`, `AGE`, `FAMSIZE`, `NCHILD`, `UHRSWORK`, `YRMARR`, `RENT`, `VALUEH`, `POVERTY`) are not categorical, so their numeric column is kept.

In [11]:
CLEAN_CHUNK_SIZE = 500_000

print(f"Loading: {DECODED_FILE}")
start = time.time()

all_columns = pd.read_csv(DECODED_FILE, nrows=0).columns.tolist()

# Numeric variables that IPUMS also labels (e.g. top-codes, N/A).
# These are real numbers, not categories, so the number is kept.
numeric_variables = {
    "YEAR",
    "AGE",
    "FAMSIZE",
    "NCHILD",
    "UHRSWORK",
    "YRMARR",
    "RENT",
    "VALUEH",
    "POVERTY",
}

# For categorical features, drop the numeric code and keep the
# readable label, e.g. DIVINYR (dropped) / DIVINYR_LABEL (kept).
drop_columns = [
    col for col in all_columns
    if f"{col}_LABEL" in all_columns
    and col not in numeric_variables
]
keep_columns = [
    col for col in all_columns
    if col not in drop_columns
]

print(f"Dropping {len(drop_columns)} columns...")

# File is several GB, so read only the kept columns and write in chunks.
# dtype=str preserves the original formatting (e.g. leading zeros).
total_rows = 0
reader = pd.read_csv(
    DECODED_FILE,
    usecols=keep_columns,
    dtype=str,
    keep_default_na=False,
    chunksize=CLEAN_CHUNK_SIZE,
)

for i, chunk in enumerate(reader):
    chunk = chunk[keep_columns]
    chunk.to_csv(
        CLEANED_FILE,
        index=False,
        mode="w" if i == 0 else "a",
        header=(i == 0),
    )
    total_rows += len(chunk)
    print(f"  processed {total_rows:,} rows...")

elapsed = time.time() - start

print()
print("========================================")
print("Cleaning complete")
print("========================================")
print(f"Rows:              {total_rows:,}")
print(f"Columns remaining: {len(keep_columns):,}")
print(f"Runtime:           {elapsed:.1f} seconds")

if CLEANED_FILE.exists():
    size_mb = CLEANED_FILE.stat().st_size / (1024 ** 2)
    print(f"Output size:       {size_mb:.1f} MB")

print()
print("Remaining columns:")

for col in keep_columns:
    print(f"  - {col}")

Loading: c:\Users\itdev\Documents\cmpt455\divorce-analysis\usa_00008_decoded.csv
Dropping 25 columns...
  processed 500,000 rows...
  processed 1,000,000 rows...
  processed 1,500,000 rows...
  processed 2,000,000 rows...
  processed 2,500,000 rows...
  processed 3,000,000 rows...
  processed 3,500,000 rows...
  processed 4,000,000 rows...
  processed 4,500,000 rows...
  processed 5,000,000 rows...
  processed 5,500,000 rows...
  processed 5,641,835 rows...

Cleaning complete
Rows:              5,641,835
Columns remaining: 54
Runtime:           127.7 seconds
Output size:       3358.4 MB

Remaining columns:
  - YEAR
  - YEAR_LABEL
  - SAMPLE_LABEL
  - SERIAL
  - CBSERIAL
  - HHWT
  - HHTYPE_LABEL
  - CLUSTER
  - REGION_LABEL
  - STATEFIP_LABEL
  - METRO_LABEL
  - STRATA
  - GQ_LABEL
  - OWNERSHP_LABEL
  - OWNERSHPD_LABEL
  - RENT
  - RENT_LABEL
  - HHINCOME
  - VALUEH
  - VALUEH_LABEL
  - COUPLETYPE_LABEL
  - PERNUM
  - PERWT
  - FAMSIZE
  - FAMSIZE_LABEL
  - NCHILD
  - NCHILD_LABEL
  -

In [12]:
# Preview of the cleaned dataset
cleaned_preview = pd.read_csv(CLEANED_FILE, nrows=10)
print(f"Cleaned columns: {cleaned_preview.shape[1]}")
cleaned_preview

Cleaned columns: 54


,YEAR,YEAR_LABEL,SAMPLE_LABEL,SERIAL,CBSERIAL,HHWT,HHTYPE_LABEL,CLUSTER,REGION_LABEL,STATEFIP_LABEL,METRO_LABEL,STRATA,GQ_LABEL,OWNERSHP_LABEL,OWNERSHPD_LABEL,RENT,RENT_LABEL,HHINCOME,VALUEH,VALUEH_LABEL,COUPLETYPE_LABEL,PERNUM,PERWT,FAMSIZE,FAMSIZE_LABEL,NCHILD,NCHILD_LABEL,SEX_LABEL,AGE,AGE_LABEL,MARST_LABEL,MARRNO_LABEL,YRMARR,YRMARR_LABEL,DIVINYR_LABEL,RACE_LABEL,RACED_LABEL,EDUC_LABEL,EDUCD_LABEL,EMPSTAT_LABEL,EMPSTATD_LABEL,LABFORCE_LABEL,CLASSWKR_LABEL,CLASSWKRD_LABEL,WKSWORK2_LABEL,UHRSWORK,UHRSWORK_LABEL,INCTOT,FTOTINC,INCWAGE,INCEARN,POVERTY,MIGRATE1_LABEL,MIGRATE1D_LABEL
0,2008,2008,2008 ACS,1,23,57.0,Married-couple family household,2008000000011,East South Central Div.,Alabama,In metropolitan area: Central/principal city s...,130001,Households under 1970 definition,Owned or being bought (loan),Owned with mortgage or loan,0,NaN,58100,210000,NaN,NaN,1,55.0,4,4,2,2,Male,37,37,"Married, spouse present",Married once,1998,NaN,Blank (No),Other Asian or Pacific Islander,Korean,5+ years of college,Master's degree,Employed,At work,"Yes, in the labor force",Works for wages,State govt employee,50-52 weeks,40,40.0,58100,58100,57000,57000,271,Same house,Same house
1,2008,2008,2008 ACS,1,23,57.0,Married-couple family household,2008000000011,East South Central Div.,Alabama,In metropolitan area: Central/principal city s...,130001,Households under 1970 definition,Owned or being bought (loan),Owned with mortgage or loan,0,NaN,58100,210000,NaN,NaN,2,49.0,4,4,2,2,Female,36,36,"Married, spouse present",Married once,1998,NaN,Blank (No),Other Asian or Pacific Islander,Korean,4 years of college,Bachelor's degree,Not in labor force,Not in Labor Force,"No, not in the labor force",NaN,NaN,NaN,0,NaN,0,58100,0,0,271,Same house,Same house
2,2008,2008,2008 ACS,2,39,113.0,Married-couple family household,2008000000021,East South Central Div.,Alabama,In metropolitan area: Central/principal city s...,90101,Households under 1970 definition,Owned or being bought (loan),Owned with mortgage or loan,0,NaN,125000,230000,NaN,NaN,1,109.0,2,2 family members present,0,0 children present,Male,30,30,"Married, spouse present",Married once,2006,NaN,Blank (No),White,White,5+ years of college,Master's degree,Employed,At work,"Yes, in the labor force",Works for wages,"Wage/salary, private",50-52 weeks,40,40.0,75000,125000,75000,75000,501,Same house,Same house
3,2008,2008,2008 ACS,2,39,113.0,Married-couple family household,2008000000021,East South Central Div.,Alabama,In metropolitan area: Central/principal city s...,90101,Households under 1970 definition,Owned or being bought (loan),Owned with mortgage or loan,0,NaN,125000,230000,NaN,NaN,2,94.0,2,2 family members present,0,0 children present,Female,30,30,"Married, spouse present",Married once,2006,NaN,Blank (No),White,White,4 years of college,Bachelor's degree,Employed,At work,"Yes, in the labor force",Works for wages,"Wage/salary, private",50-52 weeks,40,40.0,50000,125000,50000,50000,501,Same house,Same house
4,2008,2008,2008 ACS,3,70,101.0,"Female householder, no husband present",2008000000031,East South Central Div.,Alabama,In metropolitan area: Central/principal city s...,90401,Households under 1970 definition,Owned or being bought (loan),Owned free and clear,0,NaN,50000,113000,NaN,NaN,1,101.0,3,3,2,2,Female,40,40,Separated,Married once,2001,NaN,Blank (No),Black/African American,Black/African American,2 years of college,"Associate's degree, type not specified",Employed,At work,"Yes, in the labor force",Works for wages,"Wage/salary, private",50-52 weeks,40,40.0,50000,50000,50000,50000,294,Same house,Same house
5,2008,2008,2008 ACS,4,122,83.0,Married-couple family household,2008000000041,East South Central Div.,Alabama,Not in metropolitan area,110001,Households under 1970 definition,Owned or being bought (loan),Owned with mortgage or loan,0,NaN,45480,120000,NaN,NaN,1,83.0,3,3,1,1 child present,Male,55,55,"Married, spouse present",Married twice (or more),1974,NaN,Blank (No),White,White,2 years of college,"Associate'

In [13]:
# Sanity check: row counts per survey year and recent-divorce indicator
summary = pd.read_csv(CLEANED_FILE, usecols=["YEAR", "DIVINYR_LABEL"], dtype=str, keep_default_na=False)
pd.crosstab(summary["DIVINYR_LABEL"], summary["YEAR"], margins=True)

YEAR,2008,2016,2024,All
DIVINYR_LABEL,,,,
Blank (No),1761626,1825038,1995424,5582088
Yes,22428,19600,17719,59747
All,1784054,1844638,2013143,5641835
